# Traffic Volume Prediction
## Data Pre-Processing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

data = pd.read_csv('../data/Metro_Interstate_Traffic_Volume.csv')
columns = ['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'date_time', 'traffic_volume']
data = data[columns]
data.shape

(48204, 6)

In [2]:
data['date_time'] = pd.to_datetime(data['date_time'])
# sort the whole table by time so neighboring rows are neighboring hours
data = data.sort_values('date_time').reset_index(drop=True)

### Temperature

In [3]:
data.loc[data['temp'] == 0, 'temp'] = np.nan
data['temp'] = data['temp'].interpolate()

print(data['temp'].isna().sum())
print(data['temp'].min())

# convert K to F for readability
data['temp_f'] = (data['temp'] - 273.15) * 9/5 + 32

0
243.39


*Replaced temp = 0 K values with linear interpolation after sorting by time new min is 243.39 K Added temp_f in Fahrenheit for readability.*

### Data Gap

In [4]:
gaps = data['date_time'].diff()
data.loc[gaps.idxmax(), 'date_time'], gaps.max()

(Timestamp('2015-06-11 20:00:00'), Timedelta('307 days 19:00:00'))

*The data has a 307 day gap from August 2014 to June 2015*

In [5]:
after_gap = data.loc[gaps.idxmax(), 'date_time']
data = data[data['date_time'] >= after_gap].reset_index(drop=True)

print('Starting at:', after_gap)
print(len(data), 'rows remaining')

Starting at: 2015-06-11 20:00:00
32233 rows remaining


*Kept only data after the gap starting 2015-06-11 20:00 this half is larger and more recent and it avoids imputing 10 months of the target variable we may lose some accuracy but the scope becomes more manageable.*

### Duplicate Timestamps

In [6]:
dupes = data[data['date_time'].duplicated(keep=False)]
dupes.head(10)

,temp,rain_1h,snow_1h,clouds_all,date_time,traffic_volume,temp_f
53,298.27,0.55,0.0,75,2015-06-26 17:00:00,5319,77.216
54,298.27,0.55,0.0,75,2015-06-26 17:00:00,5319,77.216
55,296.23,0.55,0.0,40,2015-06-26 18:00:00,4416,73.544
56,296.23,0.55,0.0,40,2015-06-26 18:00:00,4416,73.544
57,295.88,1.02,0.0,40,2015-06-26 19:00:00,3612,72.914
58,295.88,1.02,0.0,40,2015-06-26 19:00:00,3612,72.914
66,288.08,0.60,0.0,1,2015-06-27 03:00:00,452,58.874
67,288.08,0.60,0.0,1,2015-06-27 03:00:00,452,58.874
71,289.03,0.00,0.0,1,2015-06-27 07:00:00,2048,60.584
72,289.03,0.00,0.0,1,2015-06-27 07:00:00,2048,60.584


In [7]:
before = len(data)
data = data.drop_duplicates(subset='date_time', keep='first').reset_index(drop=True)

print(before - len(data), 'duplicate rows removed')
print(data['date_time'].duplicated().sum())

5556 duplicate rows removed
0


*Removed 5,556 duplicate timestamps*

### Hourly Timeline

In [8]:
real_rows = len(data)
data = data.set_index('date_time').asfreq('h').reset_index()

missing = data['traffic_volume'].isna().sum()
print(len(data), 'total hours')
print(missing, 'missing hours', f'({missing / len(data):.1%})')

28972 total hours
2295 missing hours (7.9%)


*Reindexed to a hourly timeline so missing hours appear as NaN instead of being skipped. This keeps future lag features from pointing at the wrong hours. 2295 hours (7.9%) are missing and will be handled before modeling.*

### Rain 1H

In [9]:
data['rain_1h'].describe()


count    26677.000000
mean         0.432766
std         60.195656
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       9831.300000
Name: rain_1h, dtype: float64

In [10]:
data.nlargest(10, 'rain_1h')[['date_time', 'rain_1h']]

,date_time,rain_1h
9501,2016-07-11 17:00:00,9831.30
10365,2016-08-16 17:00:00,31.75
748,2015-07-13 00:00:00,27.57
1585,2015-08-16 21:00:00,25.46
583,2015-07-06 03:00:00,23.80
1115,2015-07-28 07:00:00,20.24
10538,2016-08-23 22:00:00,20.07
10225,2016-08-10 21:00:00,18.42
10067,2016-08-04 07:00:00,16.51
11233,2016-09-21 21:00:00,15.75


In [11]:
#mm in one hour, unrealistic above 100
threshold = 100
outliers = data['rain_1h'] > threshold

print(data.loc[outliers, ['date_time', 'rain_1h']])
data.loc[outliers, 'rain_1h'] = 0

               date_time  rain_1h
9501 2016-07-11 17:00:00   9831.3


In [12]:
data['rain_1h'].describe()

count    26677.000000
mean         0.064235
std          0.652739
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         31.750000
Name: rain_1h, dtype: float64

*The max is now a more realistic number*

### Snow

In [13]:
data['snow_1h'].describe()

count    26677.000000
mean         0.000178
std          0.007000
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.510000
Name: snow_1h, dtype: float64

In [14]:
#count hours of snow
snowy = data['snow_1h'] > 0
print(snowy.sum(), 'hours with snow')
print(f'{snowy.mean():.2%} of all hours')

31 hours with snow
0.11% of all hours


In [15]:
data.loc[snowy, ['date_time', 'snow_1h']]

,date_time,snow_1h
4672,2015-12-23 12:00:00,0.51
4674,2015-12-23 14:00:00,0.32
4676,2015-12-23 16:00:00,0.32
4678,2015-12-23 18:00:00,0.06
4740,2015-12-26 08:00:00,0.05
4741,2015-12-26 09:00:00,0.21
4742,2015-12-26 10:00:00,0.06
4798,2015-12-28 18:00:00,0.06
4801,2015-12-28 21:00:00,0.25
4802,2015-12-28 22:00:00,0.44


In [16]:
data = data.drop(columns='snow_1h')
data.columns

Index(['date_time', 'temp', 'rain_1h', 'clouds_all', 'traffic_volume',
       'temp_f'],
      dtype='str')

*snow_1h is nonzero in only 31 hours (0.11% of all hours) so the column doesn't reliably capture snow. Dropped it from the analysis.*

### Clouds

In [17]:
data['clouds_all'].describe()

count    26677.000000
mean        41.365821
std         39.065801
min          0.000000
25%          1.000000
50%         40.000000
75%         90.000000
max        100.000000
Name: clouds_all, dtype: float64

*clouds_all is cloud cover as a percentage*

In [18]:
#this shows the most common values
data['clouds_all'].value_counts().head(10)

clouds_all
1.0     9278
90.0    7227
75.0    3144
40.0    2788
5.0     1020
20.0     967
0.0      948
92.0     281
8.0      133
12.0     125
Name: count, dtype: int64

*valid range 0-100 no cleaning needed*

### Traffic Volume

In [19]:
data['traffic_volume'].describe()

count    26677.000000
mean      3293.983431
std       1969.009878
min          0.000000
25%       1260.000000
50%       3468.000000
75%       4931.000000
max       7280.000000
Name: traffic_volume, dtype: float64

In [20]:
data.nsmallest(15, 'traffic_volume')[['date_time', 'traffic_volume']]

,date_time,traffic_volume
9790,2016-07-23 18:00:00,0.0
9795,2016-07-23 23:00:00,0.0
1044,2015-07-25 08:00:00,1.0
1045,2015-07-25 09:00:00,1.0
9459,2016-07-09 23:00:00,1.0
9791,2016-07-23 19:00:00,1.0
9794,2016-07-23 22:00:00,1.0
10899,2016-09-07 23:00:00,1.0
10903,2016-09-08 03:00:00,1.0
9784,2016-07-23 12:00:00,2.0


*there are some near zero readings at times when traffic should be busy*

In [21]:
zeros = data['traffic_volume'] == 0
print(zeros.sum(), 'hours with zero traffic')
data.loc[zeros, ['date_time', 'traffic_volume']]

2 hours with zero traffic


,date_time,traffic_volume
9790,2016-07-23 18:00:00,0.0
9795,2016-07-23 23:00:00,0.0


*A busy interstate almost never has exactly 0 cars in an hour*

In [22]:
data.nlargest(10, 'traffic_volume')[['date_time', 'traffic_volume']]

,date_time,traffic_volume
15284,2017-03-09 16:00:00,7280.0
7547,2016-04-21 07:00:00,7260.0
7571,2016-04-22 07:00:00,7241.0
24860,2018-04-12 16:00:00,7213.0
7523,2016-04-20 07:00:00,7192.0
14948,2017-02-23 16:00:00,7154.0
7667,2016-04-26 07:00:00,7149.0
25148,2018-04-24 16:00:00,7149.0
25196,2018-04-26 16:00:00,7139.0
7883,2016-05-05 07:00:00,7130.0


*these times fall within weekday hours so it is valid data*

In [23]:
data.loc[zeros, 'traffic_volume'] = np.nan

*traffic_volume ranges from 0 to 7280 vehicles/hour, the highest values occur during weekday rush hours.*

*July 23, 2016 shows almost zero traffic for most of the day*

In [24]:
for t in [10, 25, 50, 100, 200]:
    print(t, (data['traffic_volume'] < t).sum())

10 31
25 40
50 41
100 44
200 61


*The count of readings plateau around 25 and 100, so we chose 50 as the cutoff between weird readings and real traffic.*

In [25]:
threshold = 50
low = data['traffic_volume'] < threshold

print(low.sum(), 'hours set to NaN')
data.loc[low, 'traffic_volume'] = np.nan

41 hours set to NaN


In [26]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 28972 entries, 0 to 28971
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date_time       28972 non-null  datetime64[us]
 1   temp            26677 non-null  float64       
 2   rain_1h         26677 non-null  float64       
 3   clouds_all      26677 non-null  float64       
 4   traffic_volume  26634 non-null  float64       
 5   temp_f          26677 non-null  float64       
dtypes: datetime64[us](1), float64(5)
memory usage: 1.3 MB


### Save Cleaned Data

In [27]:
data.to_csv('../data/traffic_clean.csv', index=False)
print('Saved', data.shape)

Saved (28972, 6)
